# Decision and episode telemetry

Executable definitions and evidence checks. Historical evidence is identified separately from new runs.


In [1]:
from pathlib import Path
import importlib.abc, importlib.util, json, sys
TRACE_ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p/'configs/notebook_execution.yaml').exists())
MODULES = {'contract':'runtime_contract','baseline':'environment_adapter','runtime':'unity_runtime'}
class NotebookModules(importlib.abc.MetaPathFinder, importlib.abc.Loader):
    def find_spec(self, fullname, path=None, target=None):
        name = MODULES.get(fullname, fullname)
        candidate = TRACE_ROOT/'notebooks/library'/f'{name}.ipynb'
        if '.' not in fullname and candidate.is_file():
            return importlib.util.spec_from_loader(fullname, self, origin=str(candidate))
    def create_module(self, spec): return None
    def exec_module(self, module):
        path=Path(module.__spec__.origin)
        module.__file__=str(path); module.TRACE_ROOT=TRACE_ROOT
        for i,cell in enumerate(json.loads(path.read_text())['cells']):
            if cell['cell_type']=='code' and 'module' in cell.get('metadata',{}).get('tags',[]):
                exec(compile(''.join(cell['source']),str(path)+f':cell-{i+1}','exec'),module.__dict__)
if not any(type(x).__name__=='NotebookModules' for x in sys.meta_path):
    sys.meta_path.insert(0,NotebookModules())
print('Notebook module loader ready:', TRACE_ROOT.name)


Notebook module loader ready: trace-lab


In [2]:
from pathlib import Path
import json,time
import numpy as np
import gymnasium as gym
from contract import write_json
class Recorder(gym.Wrapper):
    def __init__(self,env,directory):
        super().__init__(env);self.directory=Path(directory);self.events=(self.directory/'events.jsonl').open('w',buffering=1)
        self.started=time.perf_counter();self.decisions=0;self.samples=[];self.sample_ids=[];self.summaries=[];self.current=None;self.finished=False
    def emit(self,value):self.events.write(json.dumps(value,allow_nan=False)+'\n')
    def reset(self,**kwargs):
        obs,info=self.env.reset(**kwargs)
        if not self.observation_space.contains(obs):raise ValueError('Reset observation violates schema')
        self.current={'episode':self.env.state.episode,'decisions':0,'task_return':0.,'delivered_return':0.,'pairs':0,'raw_score':0.,'terminated':False,'truncated':False}
        self.samples.append(obs.copy());self.sample_ids.append({'episode':self.env.state.episode,'decision':0,'global_decision':self.decisions})
        self.emit({'event':'reset','global_decision':self.decisions,'episode':self.env.state.episode,'observation':obs.tolist(),'info':info})
        return obs,info
    def step(self,action):
        obs,reward,terminated,truncated,info=self.env.step(action);self.decisions+=1
        if not self.observation_space.contains(obs) or obs.dtype!=np.float32 or not np.isfinite(reward):raise ValueError('Invalid step')
        self.emit({'event':'decision','global_decision':self.decisions,'action':np.asarray(action).tolist(),'observation':obs.tolist(),
            'wall_seconds':time.perf_counter()-self.started,**info})
        if self.decisions%128==0:
            self.samples.append(obs.copy());self.sample_ids.append({'episode':info['episode'],'decision':info['decision'],'global_decision':self.decisions})
        self.current.update({'decisions':info['decision'],'raw_score':info['raw_score_signal'],'terminated':terminated,'truncated':truncated})
        self.current['task_return']+=info['b'];self.current['delivered_return']+=float(reward);self.current['pairs']+=int(info['fresh'])
        if terminated or truncated:
            self.current['end_reason']=info['episode_end_reason'];self.summaries.append(self.current);self.current=None
        # SB3 reserves info['episode'] for Monitor's terminal summary dictionary.
        # Preserve the adapter's integer identifier in raw evidence above, while
        # exposing it under an unreserved key at the learning/evaluation boundary.
        forwarded_info=dict(info)
        forwarded_info['episode_id']=forwarded_info.pop('episode')
        return obs,reward,terminated,truncated,forwarded_info
    def finish(self):
        if self.finished:return
        if self.current is not None and self.current['decisions']:
            self.current['end_reason']='collection_budget_fragment';self.summaries.append(self.current)
        write_json(self.directory/'episode_summaries.json',self.summaries)
        write_json(self.directory/'sample_ids.json',self.sample_ids)
        np.save(self.directory/'sample_observations.npy',np.asarray(self.samples,dtype=np.float32))
        self.events.close();self.finished=True

TelemetryRecorder=Recorder
print('Decision and episode telemetry definitions/execution completed.')


Frozen runtime contract definitions/execution completed.
Decision and episode telemetry definitions/execution completed.


Gym has been unmaintained since 2022 and does not support NumPy 2.0 amongst other critical functionality.
Please upgrade to Gymnasium, the maintained drop-in replacement of Gym, or contact the authors of your software and request that they upgrade.
Users of this version of Gym should be able to simply replace 'import gym' with 'import gymnasium as gym' in the vast majority of cases.
See the migration guide at https://gymnasium.farama.org/introduction/migration_guide/ for additional information.
